# 🚀 Mini Project 1 — Multi-Tool AI Assistant
## Google Colab | LangChain | Search | Weather | PDF RAG | Chat History | LangSmith

**Goal:** Build ONE AI agent that receives a question and automatically chooses:

  **Web Search** → latest/current/recent information

  **Weather** → live weather for a city

  **PDF RAG** → questions about an uploaded PDF

  **Chat History** → only the latest 10 message objects

  **LangSmith** → trace the agent and tool calls


### Architecture

```text
                         USER QUESTION
                                │
                                ▼
                     ┌────────────────────┐
                     │    LLM / AGENT     │
                     │   Tool Selection   │
                     └─────────┬──────────┘
                               │
              ┌────────────────┼────────────────┐
              ▼                ▼                ▼
        🔎 WEB SEARCH      🌤️ WEATHER       📄 PDF RAG
          SerpAPI          OpenWeather       FAISS
              │                │                │
              └────────────────┼────────────────┘
                               ▼
                         TOOL RESULT
                               │
                               ▼
                         FINAL ANSWER

                    + Latest 10 messages
                    + LangSmith tracing
```

# 1. 🧑‍🏫 Development Approach


### Step 1 — LLM Calling
First test a normal LLM call.

### Step 2 — Tools
Build and test Search, Weather, and PDF RAG independently.

### Step 3 — Agent
Give the three tools to one LangChain agent.

### Step 4 — Tool Calling
Understand how the LLM selects a tool and Python executes it.

### Step 5 — Memory
Keep only the latest 10 message objects.

### Step 6 — Tracing
Enable LangSmith and inspect the agent/tool execution.

### Step 7 — End-to-End Tests
Run all assignment-required questions.


# 2. Install Required Packages

We use:

- `langchain` — agent framework
- `langchain-openai` — LLM connection
- `langchain-community` — FAISS and embeddings
- `faiss-cpu` — vector database
- `sentence-transformers` — embeddings
- `pypdf` — PDF extraction
- `requests` — API calls
- `langsmith` — tracing

In [137]:
# Install packages
!pip install -q -U langchain langchain-openai langchain-community langchain-text-splitters langsmith faiss-cpu sentence-transformers pypdf requests
!pip install -q --force-reinstall requests==2.32.4

print("✅ Packages installed")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-community 0.4.2 requires requests<3.0.0,>=2.32.5, but you have requests 2.32.4 which is incompatible.
✅ Packages installed


# 3. 🔐 Secure API Keys

Do **not** hard-code API keys in the notebook.



### Required keys

| Secret | Purpose |
|---|---|
| `OPENROUTER_API_KEY` | LLM |
| `SERPAPI_API_KEY` | Web Search |
| `OPENWEATHER_API_KEY` | Live Weather |
| `LANGSMITH_API_KEY` | LangSmith |


In [138]:
# Securely load API keys
import os
from getpass import getpass

def get_secret(name):
    value = os.environ.get(name)

    if value:
        return value

    try:
        from google.colab import userdata
        value = userdata.get(name)
        if value:
            os.environ[name] = value
            return value
    except Exception:
        pass

    value = getpass(f"Enter {name}: ")
    os.environ[name] = value
    return value


for key in [
    "OPENROUTER_API_KEY",
    "SERPAPI_API_KEY",
    "OPENWEATHER_API_KEY",
    "LANGSMITH_API_KEY",
]:
    get_secret(key)

os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "Multi-Tool-AI-Assistant"

print("✅ API keys loaded securely")
print("✅ LangSmith tracing:", os.environ["LANGSMITH_TRACING"])
print("✅ LangSmith project:", os.environ["LANGSMITH_PROJECT"])

✅ API keys loaded securely
✅ LangSmith tracing: true
✅ LangSmith project: Multi-Tool-AI-Assistant


# 4. 🧠 LLM Calling

Before building an agent, test a normal LLM call.

### What is an LLM call?

```text
Python
  ↓
LLM API
  ↓
Model
  ↓
Response
```

At this stage there are **no tools**.


In [139]:
# Create the LLM
from langchain_openai import ChatOpenAI

MODEL_NAME = "openai/gpt-4o-mini"

llm = ChatOpenAI(
    model=MODEL_NAME,
    temperature=0,
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
)

print("✅ LLM initialized:", MODEL_NAME)

✅ LLM initialized: openai/gpt-4o-mini


In [140]:
# Simple LLM call
response = llm.invoke(
    "Explain AI agents in two simple sentences."
)

print(response.content)

AI agents are computer programs designed to perform tasks autonomously by perceiving their environment and making decisions based on that information. They can learn from experience and adapt their behavior to improve their performance over time.


# 5. 🔎 Tool 1 — Web Search

### When should Search be used?

Use Search for:

- latest information
- current information
- recent developments
- news
- time-sensitive questions

### Flow

```text
Agent
  ↓
web_search(query)
  ↓
SerpAPI
  ↓
Search results
  ↓
Agent
```

In [141]:
# Web Search implementation
import requests
from langchain_core.tools import tool

def web_search_impl(query: str) -> str:
    api_key = os.environ.get("SERPAPI_API_KEY")

    if not api_key:
        return "ERROR: SERPAPI_API_KEY is missing."

    try:
        r = requests.get(
            "https://serpapi.com/search.json",
            params={
                "engine": "google",
                "q": query,
                "api_key": api_key,
            },
            timeout=20,
        )

        if r.status_code != 200:
            return f"ERROR: Search API returned HTTP {r.status_code}."

        results = r.json().get("organic_results", [])

        if not results:
            return "ERROR: No search results were available."

        output = []

        for item in results[:5]:
            output.append(
                f"Title: {item.get('title', 'N/A')}\n"
                f"Snippet: {item.get('snippet', 'N/A')}\n"
                f"Link: {item.get('link', 'N/A')}"
            )

        return "\n\n".join(output)

    except Exception as e:
        return f"ERROR: Search failed — {e}"


@tool
def web_search(query: str) -> str:
    """Search the web for latest, current, recent, news, or time-sensitive information."""
    return web_search_impl(query)

print(web_search.name)
print(web_search.description)

web_search
Search the web for latest, current, recent, news, or time-sensitive information.


In [142]:
# Test Search directly
print(
    web_search.invoke({
        "query": "latest developments in generative AI"
    })
)

Title: 30+ Generative AI Trends Reshaping Business in 2026-2027
Snippet: Explore Generative AI trends for 2026 and beyond - from AI-driven creativity to edge computing. Stay ahead with this Generative AI advancements!
Link: https://masterofcode.com/blog/generative-ai-trends

Title: Generative AI Trends For All Facets of Business
Snippet: From autonomous agents to hyperpersonalized solutions, the latest trends in generative AI are shaping the future of industries, offering unparalleled ...
Link: https://www.forrester.com/technology/generative-ai/

Title: Generative AI
Snippet: This publication is dedicated to providing in-depth coverage of the latest developments, research, and applications of generative AI.
Link: https://generativeai.pub/

Title: Artificial Intelligence (Generative) Resources: Latest News on AI
Snippet: "We give AI users and leaders at enterprises the practical examples, tools and knowledge they need to thrive in the fast-changing AI era.
Link: https://guides.library.g

# 6. 🌤️ Tool 2 — Live Weather

### Purpose

Use Weather for:

- current temperature
- humidity
- wind speed
- weather condition
- city weather


In [143]:
# Weather implementation
def weather_impl(city: str) -> str:
    api_key = os.environ.get("OPENWEATHER_API_KEY")

    if not api_key:
        return "ERROR: OPENWEATHER_API_KEY is missing."

    try:
        r = requests.get(
            "https://api.openweathermap.org/data/2.5/weather",
            params={
                "q": city,
                "appid": api_key,
                "units": "metric",
            },
            timeout=20,
        )

        if r.status_code == 404:
            return f"ERROR: City '{city}' was not found."

        if r.status_code != 200:
            return f"ERROR: Weather API returned HTTP {r.status_code}."

        data = r.json()

        return (
            f"City: {data.get('name', city)}\n"
            f"Temperature: {data['main']['temp']} °C\n"
            f"Feels Like: {data['main']['feels_like']} °C\n"
            f"Humidity: {data['main']['humidity']}%\n"
            f"Wind Speed: {data.get('wind', {}).get('speed', 'N/A')} m/s\n"
            f"Condition: {data['weather'][0]['description']}"
        )

    except Exception as e:
        return f"ERROR: Weather request failed — {e}"


@tool
def get_current_weather(city: str) -> str:
    """Get current temperature, humidity, wind speed, and weather condition for a city."""
    return weather_impl(city)

print(get_current_weather.name)
print(get_current_weather.description)

get_current_weather
Get current temperature, humidity, wind speed, and weather condition for a city.


In [144]:
# Test Weather directly
print(
    get_current_weather.invoke({
        "city": "Hyderabad"
    })
)

City: Hyderabad
Temperature: 24.17 °C
Feels Like: 24.97 °C
Humidity: 89%
Wind Speed: 8.232 m/s
Condition: overcast clouds


# 7. 📄 Tool 3 — PDF RAG

### RAG = Retrieval-Augmented Generation

```text
PDF
 ↓
Extract text
 ↓
Split text
 ↓
Create embeddings
 ↓
FAISS vector database
 ↓
Similarity search
 ↓
Relevant chunks
 ↓
LLM answer
```

In [145]:
# Upload PDF
from google.colab import files
from pypdf import PdfReader

uploaded = files.upload()

pdf_name = next(
    (name for name in uploaded if name.lower().endswith(".pdf")),
    None,
)

if not pdf_name:
    raise ValueError("Please upload a PDF file.")

reader = PdfReader(pdf_name)

full_text = "\n".join(
    page.extract_text() or ""
    for page in reader.pages
)

if not full_text.strip():
    raise ValueError("No readable text was extracted from the PDF.")

print("✅ PDF:", pdf_name)
print("✅ Pages:", len(reader.pages))

Saving Shiva_Prasad_Reddy_Resume.pdf to Shiva_Prasad_Reddy_Resume (1).pdf
✅ PDF: Shiva_Prasad_Reddy_Resume (1).pdf
✅ Pages: 2


In [146]:
# Split PDF into chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=100,
)

documents = splitter.create_documents([full_text])

print("✅ Chunks:", len(documents))
print("\n--- Example chunk ---")
print(documents[0].page_content[:1000])

✅ Chunks: 14

--- Example chunk ---
SHIVA PRASAD REDDY BITLA
Executive Technical  Consultant  |  Customer Communications 
Management (CCM)  |  Insurance Domain Specialist 
Hyderabad, Telangana  •  +91 9573501294  •  shivaprasadreddy1113@gmail.com
P R O F E S S I O N A L  S U M M A R Y
Customer Communications Management (CCM) and Technical Consulting professional with 5+ years of experience


In [147]:
# Create embeddings and FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectorstore = FAISS.from_documents(
    documents,
    embeddings,
)

print("✅ FAISS created")
print("Stored vectors:", vectorstore.index.ntotal)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ FAISS created
Stored vectors: 14


In [148]:
# Test similarity search
results = vectorstore.similarity_search(
    "What is the main problem statement?",
    k=3,
)

for i, doc in enumerate(results, 1):
    print(f"--- Retrieved chunk {i} ---")
    print(doc.page_content[:1200])
    print()

--- Retrieved chunk 1 ---
across insurance and enterprise platforms. Proven track record of delivering 500+ communication changes and 
production releases, supporting large-scale insurance programs, brand transformation initiatives, and mission-critical 
production systems. Strong expertise in stakeholder management, Agile delivery, incident resolution, and end-to-

--- Retrieved chunk 2 ---
SHIVA PRASAD REDDY BITLA
Executive Technical  Consultant  |  Customer Communications 
Management (CCM)  |  Insurance Domain Specialist 
Hyderabad, Telangana  •  +91 9573501294  •  shivaprasadreddy1113@gmail.com
P R O F E S S I O N A L  S U M M A R Y
Customer Communications Management (CCM) and Technical Consulting professional with 5+ years of experience

--- Retrieved chunk 3 ---
T E C H N I C A L  S K I L L S
CCM & Communication Platforms:  Customer Communications Management, Document Management, Notification 
Systems, Bill Presentment
DevOps & Cloud:  Jenkins, Git, Docker, Kubernetes, AWS, Ansib

In [149]:
# PDF RAG tool
@tool
def pdf_rag(question: str) -> str:
    """Search the uploaded PDF and return relevant passages for answering a document question."""

    try:
        docs = vectorstore.similarity_search(question, k=4)

        if not docs:
            return "ERROR: No relevant information was found in the PDF."

        return "\n\n".join(
            f"[PDF Chunk {i}]\n{doc.page_content}"
            for i, doc in enumerate(docs, 1)
        )

    except Exception as e:
        return f"ERROR: PDF retrieval failed — {e}"


print(pdf_rag.name)
print(pdf_rag.description)

pdf_rag
Search the uploaded PDF and return relevant passages for answering a document question.


# 8. 🧰 Tool Definitions

The agent needs to understand **what each tool does**.


| Tool | Agent should use it for |
|---|---|
| `web_search` | Latest/current/recent web information |
| `get_current_weather` | Current weather |
| `pdf_rag` | Questions about uploaded PDF |

In [150]:
# All tools
tools = [
    web_search,
    get_current_weather,
    pdf_rag,
]

for t in tools:
    print("🔧", t.name)
    print("   ", t.description)
    print()

🔧 web_search
    Search the web for latest, current, recent, news, or time-sensitive information.

🔧 get_current_weather
    Get current temperature, humidity, wind speed, and weather condition for a city.

🔧 pdf_rag
    Search the uploaded PDF and return relevant passages for answering a document question.



# 9. 🤖 Build One Agent

Now we combine:

```text
LLM
 +
Tools
 +
System Prompt
 =
One Multi-Tool Agent
```

### System prompt responsibility

The system prompt tells the agent:

- when to use Search
- when to use Weather
- when to use PDF RAG
- how to handle follow-up questions
- not to invent live or PDF information


In [151]:
# Create the agent
from langchain.agents import create_agent

SYSTEM_PROMPT = """
You are a Multi-Tool AI Assistant.

TOOL SELECTION RULES:

1. WEB SEARCH:
Use web_search for latest, current, recent, news, or time-sensitive
information from the internet.

2. WEATHER:
Use get_current_weather for current temperature, humidity,
wind speed, weather conditions, or city weather.

3. PDF RAG:
Use pdf_rag for questions about the uploaded PDF.

4. FOLLOW-UP:
Use conversation history. If the user asks "What is the humidity there?",
identify the city from the previous weather conversation and use the
Weather tool for that same city.

5. DO NOT INVENT:
Do not invent live search/weather results or facts not found in the PDF.

6. FINAL ANSWER:
After receiving tool results, provide a clear concise answer.
"""

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=SYSTEM_PROMPT,
)

print("✅ Agent created")
print("Tools:", [t.name for t in tools])

✅ Agent created
Tools: ['web_search', 'get_current_weather', 'pdf_rag']


# 10. 🔄 Tool Calling — The Most Important Concept

When the user asks:

> **What is the current weather in Kolkata?**

the agent performs approximately:

```text
USER
 │
 ▼
LLM / AGENT
 │
 │ decides: weather tool
 ▼
get_current_weather(city="Kolkata")
 │
 ▼
Python executes function
 │
 ▼
OpenWeather API
 │
 ▼
Weather result
 │
 ▼
LLM / AGENT
 │
 ▼
Final human answer
```


# 11. 💬 Latest-10 Chat History

We implement:

```python
chat_history = result["messages"][-10:]
```

So the history never grows beyond 10 stored messages.

In [152]:
import inspect
from langchain_core.messages import ToolMessage, AIMessage, HumanMessage

# Conversation runner
chat_history = []

def safe_slice_history(history_list, max_limit=10):
    """Slices chat history strictly to max_limit without splitting an AIMessage
    tool call from its corresponding ToolMessage response.
    """
    if len(history_list) <= max_limit:
        return history_list

    # Force clip to the maximum allowed limit first
    pruned_history = history_list[-max_limit:]

    # Inspect the leading message to avoid breaking LLM API sequence constraints
    while pruned_history:
        first_msg = pruned_history[0]

        # Check role type safely across object types or dictionaries
        role = getattr(first_msg, 'role', getattr(first_msg, 'type', None))
        if role is None and isinstance(first_msg, dict):
            role = first_msg.get('role')

        # Rule 1: A ToolMessage cannot start the conversation history window
        if role == 'tool' or isinstance(first_msg, ToolMessage):
            pruned_history.pop(0)
            continue

        # Rule 2: An AIMessage with tool calls cannot start the window if its response was cut
        if role == 'assistant' or isinstance(first_msg, AIMessage):
            has_tool_calls = bool(getattr(first_msg, 'tool_calls', None) or
                                  (isinstance(first_msg, dict) and first_msg.get('tool_calls')))
            if has_tool_calls:
                pruned_history.pop(0)
                continue

        # Valid entry point discovered (typically a HumanMessage)
        break

    return pruned_history

def ask_agent(question: str):
    global chat_history

    # Clean up the history before adding the new input sequence
    sliced_history = safe_slice_history(chat_history, 10)

    # Wrap the new user request structure
    messages = sliced_history + [HumanMessage(content=question)]

    # Invoke your compiled agent framework graph
    result = agent.invoke({
        "messages": messages
    })

    # Enforce strict memory limits safely on the returned pipeline messages
    chat_history = safe_slice_history(result["messages"], 10)

    # Safely isolate and process the text output string
    final_message = chat_history[-1]
    answer = (
        final_message.content
        if hasattr(final_message, "content")
        else (final_message.get("content", str(final_message)) if isinstance(final_message, dict) else str(final_message))
    )

    print("USER")
    print(question)

    print("\n ASSISTANT")
    print(answer)

    print("\n Stored message objects:", len(chat_history))

    return answer


In [153]:
# TEST — Clear history
print("Before clearing:", len(chat_history))

clear_chat_history()

assert len(chat_history) == 0

print("✅ PASS — Chat history successfully cleared.")

Before clearing: 0
🧹 Chat history cleared.
Stored message objects: 0
✅ PASS — Chat history successfully cleared.


# 12.Test 1 — Current Web Information

### Question

> **What are the latest developments in generative AI?**

Expected tool:

```text
web_search
```


In [154]:
# TEST 1 — Search
ask_agent(
    "What are the latest developments in generative AI?"
)

USER
What are the latest developments in generative AI?

 ASSISTANT
Here are some of the latest developments in generative AI as of 2023:

1. **Widespread Adoption**: Generative AI has seen significant adoption across various sectors, with enterprises spending approximately $2.5 billion on these technologies. Tools like GitHub Copilot and Hugging Face are leading the charge in this space.

2. **Diverse Applications**: The year has marked advancements in applications such as transcription, deepfaking, voice-cloning, and language translation. Generative AI tools are increasingly being integrated into business processes, enhancing productivity and creativity.

3. **Emerging Trends**: Key trends include the rise of autonomous agents and hyperpersonalized solutions, which are transforming how businesses operate and interact with customers.

4. **Talent and Innovation**: There is a growing demand for AI-related talent, as companies strive to stay ahead in the rapidly evolving landscape of ge

'Here are some of the latest developments in generative AI as of 2023:\n\n1. **Widespread Adoption**: Generative AI has seen significant adoption across various sectors, with enterprises spending approximately $2.5 billion on these technologies. Tools like GitHub Copilot and Hugging Face are leading the charge in this space.\n\n2. **Diverse Applications**: The year has marked advancements in applications such as transcription, deepfaking, voice-cloning, and language translation. Generative AI tools are increasingly being integrated into business processes, enhancing productivity and creativity.\n\n3. **Emerging Trends**: Key trends include the rise of autonomous agents and hyperpersonalized solutions, which are transforming how businesses operate and interact with customers.\n\n4. **Talent and Innovation**: There is a growing demand for AI-related talent, as companies strive to stay ahead in the rapidly evolving landscape of generative AI.\n\nFor more detailed insights, you can explore

# 13.Test 2 — Current Weather

1.   List item
2.   List item



### Question

> **What is the current weather in Kolkata?**

Expected tool:

```text
get_current_weather
```

In [155]:
# TEST 2 — Weather
ask_agent(
    "What is the current weather in Kolkata?"
)

USER
What is the current weather in Kolkata?

 ASSISTANT
The current weather in Kolkata is as follows:
- **Temperature**: 27.97 °C (Feels like 34.36 °C)
- **Humidity**: 94%
- **Wind Speed**: 4.12 m/s
- **Condition**: Overcast clouds

 Stored message objects: 8


'The current weather in Kolkata is as follows:\n- **Temperature**: 27.97 °C (Feels like 34.36 °C)\n- **Humidity**: 94%\n- **Wind Speed**: 4.12 m/s\n- **Condition**: Overcast clouds'

# 14.Test 3 — PDF Question 1

### Question

> **According to the uploaded PDF, what is the main problem statement?**

Expected tool:

```text
pdf_rag
```

In [156]:
# TEST 3 — PDF Question 1
ask_agent(
    "According to the uploaded PDF, short summary of the pdf?"
)

USER
According to the uploaded PDF, short summary of the pdf?

 ASSISTANT
The PDF provides a professional summary of Shiva Prasad Reddy Bitla, an Executive Technical Consultant specializing in Customer Communications Management (CCM) within the insurance domain. Key points include:

- Over 5 years of experience in technical consulting and customer communications.
- Managed end-to-end requirements gathering and solution delivery for various insurance communication initiatives.
- Achievements include delivering over 500 communication changes and ensuring stable, zero-downtime production releases.
- Expertise in stakeholder management, Agile delivery, and incident resolution, with a focus on supporting large-scale insurance programs and mission-critical production systems.

The document highlights his role in managing production deployments, providing L3 support, and preparing documentation for deployment and knowledge transfer.

 Stored message objects: 9


'The PDF provides a professional summary of Shiva Prasad Reddy Bitla, an Executive Technical Consultant specializing in Customer Communications Management (CCM) within the insurance domain. Key points include:\n\n- Over 5 years of experience in technical consulting and customer communications.\n- Managed end-to-end requirements gathering and solution delivery for various insurance communication initiatives.\n- Achievements include delivering over 500 communication changes and ensuring stable, zero-downtime production releases.\n- Expertise in stakeholder management, Agile delivery, and incident resolution, with a focus on supporting large-scale insurance programs and mission-critical production systems.\n\nThe document highlights his role in managing production deployments, providing L3 support, and preparing documentation for deployment and knowledge transfer.'

# 15. Test 4 — PDF Question 2

### Question

> **According to the uploaded PDF, what are the basic requirements?**

Expected tool:

```text
pdf_rag
```

In [157]:
# TEST 4 — PDF Question 2
ask_agent(
    "According to the uploaded PDF, what are the basic requirements?"
)

USER
According to the uploaded PDF, what are the basic requirements?

 ASSISTANT
The basic requirements outlined in the PDF include:

- **Databases**: MySQL, PostgreSQL
- **Operating Systems**: Linux, Windows
- **Tools**: JIRA, Agile/Scrum, SDLC, Release Management, Production Support

Additionally, the document mentions technical skills in Customer Communications Management (CCM), DevOps, and programming languages such as Python, JavaScript, HTML, JSON, XML, and SQL.

 Stored message objects: 9


'The basic requirements outlined in the PDF include:\n\n- **Databases**: MySQL, PostgreSQL\n- **Operating Systems**: Linux, Windows\n- **Tools**: JIRA, Agile/Scrum, SDLC, Release Management, Production Support\n\nAdditionally, the document mentions technical skills in Customer Communications Management (CCM), DevOps, and programming languages such as Python, JavaScript, HTML, JSON, XML, and SQL.'

# 16.Test 5 — Follow-Up

### Question

> **What is the humidity there?**

This tests conversation context.

The agent should understand that **“there”** refers to the city from the previous weather question.

Expected:

```text
Previous question → Kolkata
Follow-up → "there"
Agent → Kolkata
Agent → Weather tool
```

In [158]:
# TEST 5 — Follow-up
ask_agent(
    "What is the humidity in there"
)

USER
What is the humidity in there

 ASSISTANT
The humidity in Kolkata is currently 94%.

 Stored message objects: 9


'The humidity in Kolkata is currently 94%.'

# 17. Test 6 — Prove Latest-10 Limit

We continue the conversation and verify that the stored history never exceeds 10 messages.

In [159]:
# TEST 6 — Latest 10 messages
for i in range(1, 8):
    ask_agent(
        f"Give me one short observation about demonstration message {i}."
    )

    assert len(chat_history) <= 10

print("\n✅ PASS — History never exceeded 10 messages.")
print("Final stored message count:", len(chat_history))

USER
Give me one short observation about demonstration message 1.

 ASSISTANT
One short observation about demonstration message 1 is that it successfully delivered over 200 taxation and customer notification enhancements, which were validated across more than 500 test scenarios, significantly improving notification delivery performance.

 Stored message objects: 9
USER
Give me one short observation about demonstration message 2.

 ASSISTANT
One short observation about demonstration message 2 is that it highlights the management of end-to-end requirements gathering and solution delivery across multiple insurance communication initiatives, ensuring stable and reliable production releases.

 Stored message objects: 9
USER
Give me one short observation about demonstration message 3.

 ASSISTANT
One short observation about demonstration message 3 is that it emphasizes the provision of L3 support for telecom and enterprise customer communication applications, showcasing a commitment to maint

# 18. 📊 LangSmith Tracing

### Why tracing?

LangSmith allows us to inspect the actual execution.

For example:

```text
User Question
     ↓
Agent
     ↓
Tool Selected
     ↓
Tool Input
     ↓
Tool Output
     ↓
Final Answer
```

In [160]:
# LangSmith configuration check
print("LANGSMITH_TRACING:", os.environ.get("LANGSMITH_TRACING"))
print("LANGSMITH_PROJECT:", os.environ.get("LANGSMITH_PROJECT"))
print("API key configured:", bool(os.environ.get("LANGSMITH_API_KEY")))

print("\n✅ Run the agent tests above.")
print("Then open LangSmith and inspect the trace.")

LANGSMITH_TRACING: true
LANGSMITH_PROJECT: Multi-Tool-AI-Assistant
API key configured: True

✅ Run the agent tests above.
Then open LangSmith and inspect the trace.


# 19. ⚠️ Error Handling

The simple errors for:

- missing API key
- invalid city
- unavailable result

Our functions return readable `ERROR:` messages.

In [161]:
# Error test — invalid city
print(
    weather_impl(
        "CityThatDefinitelyDoesNotExist999999"
    )
)

ERROR: City 'CityThatDefinitelyDoesNotExist999999' was not found.


# 21. 🏗️ Final Architecture

```text
                         ┌──────────────────────┐
                         │     USER QUESTION    │
                         └──────────┬───────────┘
                                    │
                                    ▼
                         ┌──────────────────────┐
                         │    LANGCHAIN AGENT   │
                         │         + LLM        │
                         └──────────┬───────────┘
                                    │
                 ┌──────────────────┼──────────────────┐
                 ▼                  ▼                  ▼
          ┌────────────┐     ┌────────────┐     ┌────────────┐
          │ WEB SEARCH │     │  WEATHER   │     │  PDF RAG   │
          │  SerpAPI   │     │ OpenWeather│     │ FAISS      │
          └──────┬─────┘     └──────┬─────┘     └──────┬─────┘
                 │                  │                  │
                 └──────────────────┼──────────────────┘
                                    │
                                    ▼
                         ┌──────────────────────┐
                         │     TOOL RESULTS     │
                         └──────────┬───────────┘
                                    │
                                    ▼
                         ┌──────────────────────┐
                         │     FINAL ANSWER     │
                         └──────────────────────┘

              Supporting components:
              ├── Latest 10 messages
              ├── Secure API keys
              ├── Error handling
              └── LangSmith tracing
```

In [163]:
# AGENT HEALTH CHECK
checks = {
    "LLM initialized": "llm" in globals(),
    "Search tool": "web_search" in globals(),
    "Weather tool": "get_current_weather" in globals(),
    "PDF RAG tool": "pdf_rag" in globals(),
    "Agent": "agent" in globals(),
    "FAISS vector store": "vectorstore" in globals(),
    "History <= 10": len(chat_history) <= 10,
    "LangSmith enabled": os.environ.get("LANGSMITH_TRACING") == "true",
}

print("========== AGENT HEALTH CHECK ==========")

for name, passed in checks.items():
    print(("✅" if passed else "❌"), name)

print("==========================================")

========== AGENT HEALTH CHECK ==========
✅ LLM initialized
✅ Search tool
✅ Weather tool
✅ PDF RAG tool
✅ Agent
✅ FAISS vector store
✅ History <= 10
✅ LangSmith enabled
